In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_dir = "/content/drive/MyDrive/GATv2_Datasets"

In [2]:
import torch

# Check if CUDA is available
if torch.cuda.is_available():
    CUDA = 'cu' + torch.version.cuda.replace('.', '')
else:
    # Fallback for CPU if you decide not to use GPU
    CUDA = 'cpu'

TORCH = torch.__version__.split('+')[0]

print(f"Detected PyTorch: {TORCH} and CUDA: {CUDA}")

# Install dependencies using the official PyG wheel index
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
!pip install torch-geometric

Detected PyTorch: 2.10.0 and CUDA: cu128
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 121.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 149.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 120.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 549.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.7 MB/s eta 0:00:00


In [21]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv

# 1. Load the Cora dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]


In [15]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv

# 1. Load the Cora dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

# 2. Define the GAT Network
class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        # First layer: multi-head attention (8 heads is the paper standard for Cora)
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        # Second layer: output layer (averaging heads for classification)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

# 3. Setup Training
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GAT(dataset.num_features, 8, dataset.num_classes, heads=8).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# 4. Training Loop
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss

for epoch in range(200):
    loss = train()
    if epoch % 20 == 0:
        print(f'Epoch {epoch:03d}, Loss: {loss:.4f}')

# 5. Evaluate
model.eval()
pred = model(data.x, data.edge_index).argmax(dim=1)
acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
print(f'Test Accuracy: {acc:.4f}')

Epoch 000, Loss: 1.9815
Epoch 020, Loss: 0.9211
Epoch 040, Loss: 0.6519
Epoch 060, Loss: 0.5182
Epoch 080, Loss: 0.4885
Epoch 100, Loss: 0.4426
Epoch 120, Loss: 0.4044
Epoch 140, Loss: 0.3881
Epoch 160, Loss: 0.4190
Epoch 180, Loss: 0.3908
Test Accuracy: 0.7970


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GATLayer(nn.Module):
    def __init__(self, in_features, out_features, dropout, alpha, concat=True, gain=1.414):
        super(GATLayer, self).__init__()
        self.dropout = dropout
        self.in_features = in_features
        self.out_features = out_features
        self.alpha = alpha
        self.concat = concat

        # 1. W initialization using the passed gain
        self.W = nn.Parameter(torch.empty(size=(in_features, out_features)))
        nn.init.xavier_uniform_(self.W.data, gain=gain)

        # 2. a initialization using the passed gain
        self.a = nn.Parameter(torch.empty(size=(2 * out_features, 1)))
        nn.init.xavier_uniform_(self.a.data, gain=gain)

        self.leakyrelu = nn.LeakyReLU(self.alpha)

    def forward(self, h, adj):
        Wh = torch.mm(h, self.W)
        N = Wh.size()[0]

        # Compute Attention
        a_input = torch.cat([Wh.repeat_interleave(N, dim=0), Wh.repeat(N, 1)], dim=1)
        a_input = a_input.view(N, N, 2 * self.out_features)
        e = self.leakyrelu(torch.matmul(a_input, self.a).squeeze(2))

        # Masked Softmax
        zero_vec = -1e9 * torch.ones_like(e) # Using -1e9 for better stability
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=1)
        attention = F.dropout(attention, self.dropout, training=self.training)

        # Aggregate
        h_prime = torch.matmul(attention, Wh)

        # 3. Logic check: Only apply ELU if concat=True (Hidden Layers)
        # If concat=False (Output Layer), return raw logits for Softmax
        if self.concat:
            return F.elu(h_prime)
        else:
            return h_prime

class GAT_Model(nn.Module):
    def __init__(self, nfeat, nhid, nclass, dropout, alpha, nheads):
        super(GAT_Model, self).__init__()
        self.dropout = dropout

        # Hidden layers: concat=True, gain=1.414 (for ELU)
        self.attentions = nn.ModuleList([
            GATLayer(nfeat, nhid, dropout=dropout, alpha=alpha, concat=True, gain=1.414)
            for _ in range(nheads)
        ])

        # Output layer: concat=False, gain=1.0 (for Softmax stability)
        self.out_att = GATLayer(nhid * nheads, nclass, dropout=dropout, alpha=alpha, concat=False, gain=1.0)

    def forward(self, x, adj):
        # Feature Dropout
        x = F.dropout(x, self.dropout, training=self.training)

        # Multi-head concat
        x = torch.cat([att(x, adj) for att in self.attentions], dim=1)

        # Second Dropout (applied to concatenated hidden features)
        x = F.dropout(x, self.dropout, training=self.training)

        # Final Layer (No ELU here, just linear + LogSoftmax)
        x = self.out_att(x, adj)

        return F.log_softmax(x, dim=1)

In [9]:
import torch.optim as optim
from torch_geometric.utils import to_dense_adj

# 1. Convert Edge Index to Dense Adjacency for your custom layer
# This is necessary because your Layer expects [N, N]
adj = to_dense_adj(data.edge_index)[0]
# Add self-loops (GAT paper assumes every node attends to itself)
adj = adj + torch.eye(adj.size(0)).to(device)

# 2. Initialize your custom model
model = GAT_Model(
    nfeat=dataset.num_features,
    nhid=8,
    nclass=dataset.num_classes,
    dropout=0.6,
    alpha=0.2,
    nheads=8
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# 3. Modified Training Loop
def train_custom():
    model.train()
    optimizer.zero_grad()
    # Notice we pass 'adj' instead of 'edge_index'
    out = model(data.x, adj)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss

# Run training
for epoch in range(200):
    loss = train_custom()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f}")

Epoch 000 | Loss: 2.1936
Epoch 020 | Loss: 0.9924
Epoch 040 | Loss: 0.7560
Epoch 060 | Loss: 0.5477
Epoch 080 | Loss: 0.4392
Epoch 100 | Loss: 0.5057
Epoch 120 | Loss: 0.4241
Epoch 140 | Loss: 0.3959
Epoch 160 | Loss: 0.4283
Epoch 180 | Loss: 0.4030


In [10]:
def test():
    model.eval()
    with torch.no_grad():
        # Pass the dense adj matrix just like in training
        logits = model(data.x, adj)

        # Get the class with the highest probability
        pred = logits.argmax(dim=1)

        # Check against test_mask
        test_correct = (pred[data.test_mask] == data.y[data.test_mask]).sum().item()
        test_acc = test_correct / data.test_mask.sum().item()

        # Optional: Check validation accuracy too
        val_correct = (pred[data.val_mask] == data.y[data.val_mask]).sum().item()
        val_acc = val_correct / data.val_mask.sum().item()

    return val_acc, test_acc

# Execute the test
val_acc, test_acc = test()
print(f'Validation Accuracy: {val_acc:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

Validation Accuracy: 0.7820
Test Accuracy: 0.8090


In [17]:
import torch.optim as optim
from torch_geometric.utils import to_dense_adj

# 1. Prepare Data
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
adj = to_dense_adj(data.edge_index)[0]
adj = adj + torch.eye(adj.size(0)) # Add Self-loops
adj = adj.to(device)
features = data.x.to(device)
labels = data.y.to(device)

# 2. Initialize Model with Paper Hyperparams
model = GAT_Model(
    nfeat=dataset.num_features,
    nhid=8,
    nclass=dataset.num_classes,
    dropout=0.6,
    alpha=0.2,
    nheads=8
).to(device)

# 3. Optimizer with specific Weight Decay
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

# Now run your training loop!

In [23]:
import copy

# Configuration
NUM_RUNS = 10
all_run_test_accs = []
absolute_best_test_acc = 0
absolute_best_state = None

print(f"Starting {NUM_RUNS} independent runs to reproduce paper results...")

for run in range(NUM_RUNS):
    # 1. RESET MODEL AND OPTIMIZER FOR EACH RUN
    # This ensures each run is independent
    model = GAT_Model(
        nfeat=dataset.num_features,
        nhid=8,
        nclass=dataset.num_classes,
        dropout=0.6,
        alpha=0.2,
        nheads=8
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

    # Reset early stopping variables for this run
    best_val_acc = 0
    run_best_model_state = None
    bad_counter = 0

    for epoch in range(MAX_EPOCHS):
        # Training Step
        model.train()
        optimizer.zero_grad()
        out = model(features, adj_gpu)
        loss = F.nll_loss(out[train_mask], labels[train_mask])
        loss.backward()
        optimizer.step()

        # Validation Step
        model.eval()
        with torch.no_grad():
            logits = model(features, adj_gpu)
            val_pred = logits.argmax(dim=1)
            val_acc = (val_pred[val_mask] == labels[val_mask]).sum().item() / val_mask.sum().item()

        # Early Stopping Logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            bad_counter = 0
            run_best_model_state = copy.deepcopy(model.state_dict())
        else:
            bad_counter += 1

        if bad_counter == PATIENCE:
            break

    # 2. EVALUATE THE BEST VERSION OF THIS RUN
    model.load_state_dict(run_best_model_state)
    model.eval()
    with torch.no_grad():
        logits = model(features, adj_gpu)
        test_pred = logits.argmax(dim=1)
        test_acc = (test_pred[test_mask] == labels[test_mask]).sum().item() / test_mask.sum().item()

    all_run_test_accs.append(test_acc)
    print(f"Run {run+1}/{NUM_RUNS} | Best Val Acc: {best_val_acc:.4f} | Test Acc: {test_acc:.4f}")

    # Track the absolute best across all runs
    if test_acc > absolute_best_test_acc:
        absolute_best_test_acc = test_acc
        absolute_best_state = run_best_model_state

# 3. FINAL SUMMARY
print("\n--- Final Summary ---")
print(f"Mean Test Accuracy: {np.mean(all_run_test_accs):.4f}")
print(f"Max Test Accuracy achieved: {absolute_best_test_acc:.4f}")

Starting 10 independent runs to reproduce paper results...
Run 1/10 | Best Val Acc: 0.7960 | Test Acc: 0.8090
Run 2/10 | Best Val Acc: 0.8100 | Test Acc: 0.8120
Run 3/10 | Best Val Acc: 0.8000 | Test Acc: 0.8060
Run 4/10 | Best Val Acc: 0.8020 | Test Acc: 0.8220
Run 5/10 | Best Val Acc: 0.7980 | Test Acc: 0.8030
Run 6/10 | Best Val Acc: 0.8000 | Test Acc: 0.8180
Run 7/10 | Best Val Acc: 0.7940 | Test Acc: 0.8060
Run 8/10 | Best Val Acc: 0.7980 | Test Acc: 0.8040
Run 9/10 | Best Val Acc: 0.7960 | Test Acc: 0.8010
Run 10/10 | Best Val Acc: 0.7940 | Test Acc: 0.7960

--- Final Summary ---
Mean Test Accuracy: 0.8077
Max Test Accuracy achieved: 0.8220


In [ ]:
import copy

# Configuration
NUM_RUNS = 10
all_run_test_accs = []
absolute_best_test_acc = 0
absolute_best_state = None

print(f"Starting {NUM_RUNS} independent runs to reproduce paper results...")

for run in range(NUM_RUNS):
    # 1. RESET MODEL AND OPTIMIZER FOR EACH RUN
    # This ensures each run is independent
    model = GAT_Model(
        nfeat=dataset.num_features,
        nhid=8,
        nclass=dataset.num_classes,
        dropout=0.6,
        alpha=0.2,
        nheads=8
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

    # Reset early stopping variables for this run
    best_val_acc = 0
    run_best_model_state = None
    bad_counter = 0

    for epoch in range(MAX_EPOCHS):
        # Training Step
        model.train()
        optimizer.zero_grad()
        out = model(features, adj_gpu)
        loss = F.nll_loss(out[train_mask], labels[train_mask])
        loss.backward()
        optimizer.step()

        # Validation Step
        model.eval()
        with torch.no_grad():
            logits = model(features, adj_gpu)
            val_pred = logits.argmax(dim=1)
            val_acc = (val_pred[val_mask] == labels[val_mask]).sum().item() / val_mask.sum().item()

        # Early Stopping Logic
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            bad_counter = 0
            run_best_model_state = copy.deepcopy(model.state_dict())
        else:
            bad_counter += 1

        if bad_counter == PATIENCE:
            break

    # 2. EVALUATE THE BEST VERSION OF THIS RUN
    model.load_state_dict(run_best_model_state)
    model.eval()
    with torch.no_grad():
        logits = model(features, adj_gpu)
        test_pred = logits.argmax(dim=1)
        test_acc = (test_pred[test_mask] == labels[test_mask]).sum().item() / test_mask.sum().item()

    all_run_test_accs.append(test_acc)
    print(f"Run {run+1}/{NUM_RUNS} | Best Val Acc: {best_val_acc:.4f} | Test Acc: {test_acc:.4f}")

    # Track the absolute best across all runs
    if test_acc > absolute_best_test_acc:
        absolute_best_test_acc = test_acc
        absolute_best_state = run_best_model_state

# 3. FINAL SUMMARY
print("\n--- Final Summary ---")
print(f"Mean Test Accuracy: {np.mean(all_run_test_accs):.4f}")
print(f"Max Test Accuracy achieved: {absolute_best_test_acc:.4f}")

Starting 10 independent runs to reproduce paper results...


In [25]:
PATIENCE = 200
MAX_EPOCHS = 2000